In [1]:
import requests
import pandas as pd
from collections import defaultdict

In [2]:
DISEASE_PREFIXES = {
    "Hypertension": ["C02", "C03", "C07", "C08", "C09"],
    "High Cholesterol": ["C10"],
    "Diabetes": ["A10"],
    "Obesity": ["A08"],
    "Arthritis": ["M01", "M02", "M04", "L04"],
}


In [3]:
def get_resp(class_id):
    """add in docstring"""

    url = f'https://rxnav.nlm.nih.gov/REST/rxclass/classMembers.json?classId={class_id}&relaSource=ATC'

    # add try and except here! 
    resp = requests.get(url)

    return resp.json()


def extract_data_mapping(atc_codes):
    drug_mapping = defaultdict(list)

    for disease, codes in atc_codes.items():
        for code in codes:
            
            drug_list = get_drugs_by_code(code)
            drug_mapping[disease].extend(drug_list)

    return drug_mapping

def get_drugs_by_code(code):
    res_list = []
    data = get_resp(code)
    names = data['drugMemberGroup']['drugMember']

    for drug in names:
        code = drug['nodeAttr'][0]['attrValue']
        drug_name = drug['nodeAttr'][1]['attrValue']
        
        res_list.append({
            'code': code,
            'drug_name': drug_name
        })
    
    return res_list



In [4]:
data = extract_data_mapping(DISEASE_PREFIXES)
data

defaultdict(list,
            {'Hypertension': [{'code': 'C02BA01', 'drug_name': 'trimetaphan'},
              {'code': 'C02KX05', 'drug_name': 'riociguat'},
              {'code': 'C02KX04', 'drug_name': 'macitentan'},
              {'code': 'C02CC01', 'drug_name': 'betanidine'},
              {'code': 'C02AA06', 'drug_name': 'methoserpidine'},
              {'code': 'C02AC01', 'drug_name': 'clonidine'},
              {'code': 'C02KX54', 'drug_name': 'macitentan and tadalafil'},
              {'code': 'C02KB01', 'drug_name': 'metirosine'},
              {'code': 'C02KX06', 'drug_name': 'sotatercept'},
              {'code': 'C02KN01', 'drug_name': 'aprocitentan'},
              {'code': 'C02AC05', 'drug_name': 'moxonidine'},
              {'code': 'C02CC04', 'drug_name': 'debrisoquine'},
              {'code': 'C02DA01', 'drug_name': 'diazoxide'},
              {'code': 'C02DG01', 'drug_name': 'pinacidil'},
              {'code': 'C02DB01', 'drug_name': 'dihydralazine'},
             

In [5]:
data['Hypertension']

[{'code': 'C02BA01', 'drug_name': 'trimetaphan'},
 {'code': 'C02KX05', 'drug_name': 'riociguat'},
 {'code': 'C02KX04', 'drug_name': 'macitentan'},
 {'code': 'C02CC01', 'drug_name': 'betanidine'},
 {'code': 'C02AA06', 'drug_name': 'methoserpidine'},
 {'code': 'C02AC01', 'drug_name': 'clonidine'},
 {'code': 'C02KX54', 'drug_name': 'macitentan and tadalafil'},
 {'code': 'C02KB01', 'drug_name': 'metirosine'},
 {'code': 'C02KX06', 'drug_name': 'sotatercept'},
 {'code': 'C02KN01', 'drug_name': 'aprocitentan'},
 {'code': 'C02AC05', 'drug_name': 'moxonidine'},
 {'code': 'C02CC04', 'drug_name': 'debrisoquine'},
 {'code': 'C02DA01', 'drug_name': 'diazoxide'},
 {'code': 'C02DG01', 'drug_name': 'pinacidil'},
 {'code': 'C02DB01', 'drug_name': 'dihydralazine'},
 {'code': 'C02KX02', 'drug_name': 'ambrisentan'},
 {'code': 'C02CA06', 'drug_name': 'urapidil'},
 {'code': 'C02AC02', 'drug_name': 'guanfacine'},
 {'code': 'C02CA04', 'drug_name': 'doxazosin'},
 {'code': 'C02CC02', 'drug_name': 'guanethidine'

In [6]:
def json_to_dfs(mapping):
    """
    Convert a dict of lists of dicts into a dict of DataFrames.

    Parameters:
        mapping (dict): 
            { condition: [ {code:..., drug_name:...}, ... ] }

    Returns:
        dict: 
            { condition: DataFrame }
    """
    df_lst = []

    for condition, records in mapping.items():
        # Each `records` is already a list of dicts → normalize directly into a DF
        df = pd.json_normalize(records)
        df['condition'] = condition
        df_lst.append(df)

    return pd.concat(df_lst, ignore_index=True)

In [7]:
test = json_to_dfs(data)
test

,code,drug_name,condition
0,C02BA01,trimetaphan,Hypertension
1,C02KX05,riociguat,Hypertension
2,C02KX04,macitentan,Hypertension
3,C02CC01,betanidine,Hypertension
4,C02AA06,methoserpidine,Hypertension
...,...,...,...
474,L04AC04,rilonacept,Arthritis
475,L04AB06,golimumab,Arthritis
476,L04AC05,ustekinumab,Arthritis
477,L04AC08,canakinumab,Arthritis


In [ ]:
test.to_csv('new_mapping.csv', index=False)